In [1]:
import pickle
import os
import sys
import numpy as np
import datetime

In [2]:
current_path = os.getcwd()
cwd = os.path.dirname(current_path)
sys.path.append(os.path.join(cwd, "utilities"))

from optimization_dataclasses import Prosumer_Cluster_Data, Market_Data, Bivariate_Pwl_Input, Layer_Data
from pickle_file_operations import read_pickle, write_pickle
from Energinet_API import energinet_api_wrapper 

In [4]:
pickle_path = os.path.join(cwd, "pickle files")
optimization_input_path = os.path.join(cwd, "optimization input")

instance_data = read_pickle(os.path.join(pickle_path, "instance.pickle"))
flexibility_scenarios = read_pickle(os.path.join(pickle_path, "flexibility_scenarios.pkl"))
prosumer_responsiveness = read_pickle(os.path.join(pickle_path, "prosumRespMultiPWL.pickle"))
CCP = read_pickle(os.path.join(pickle_path, "CCP.pickle"))

In [5]:
# generate prosumer_cluster_data

data = CCP[str(0.6)]


prosumer_cluster = {}
for key in data.keys():

    if key != "Block":
        max_flexibility_profile = [x + y for x, y in zip(data[key], data["Block"])]
        flexibility_scenario = flexibility_scenarios[key + "1"]

        cross_elasticity_matrix = flexibility_scenario["Single Order"]["Cross Elasticity"] + flexibility_scenario["Block Order"]["Cross Elasticity"]
        norm_cross_elasticity_matrix = []
        
        for elems in cross_elasticity_matrix:
            if sum(elems) != 0:
                norm_cross_elasticity_matrix.append([round(elem / sum(elems), 3) for elem in elems])
            else:
                norm_cross_elasticity_matrix.append([-elem for elem in elems])
        
        norm_cross_elasticity_matrix = np.array(norm_cross_elasticity_matrix)

        prosumer_cluster[key] = Prosumer_Cluster_Data(cross_temporal_matrix=norm_cross_elasticity_matrix, 
                                                      max_flexibility_profile=max_flexibility_profile)

write_pickle(prosumer_cluster, os.path.join(optimization_input_path, "prosumer_cluster_data.pickle"))

In [6]:
# generate PWL dataset

data_dict = prosumer_responsiveness[5]

pwl_data_input = {}

for key, data in data_dict.items():
    a, b = data["(a, b)"]
    pwl_data_input[key] = Bivariate_Pwl_Input(shaping_parameters=(a, -b), 
                                              flexibility_points=data["Flex breakpoint"], 
                                              max_flexibility_points=data["max Flex breakpoint"], 
                                              cost_points=data["Flex Cost breakpoint"]
                                             ) 

write_pickle(pwl_data_input, os.path.join(optimization_input_path, "5_pwl_data_input.pickle"))

In [7]:
prosumer_responsiveness[5]

{0: {'Flex breakpoint': [0.098, 4.974, 9.849, 14.725, 19.6],
  'Flex Cost breakpoint': array([[3.70000e+00, 1.04400e+00, 8.08000e-01, 6.71000e-01, 5.73000e-01],
         [0.00000e+00, 1.87877e+02, 1.20340e+02, 1.08561e+02, 1.01628e+02],
         [0.00000e+00, 0.00000e+00, 3.71843e+02, 2.60941e+02, 2.37946e+02],
         [0.00000e+00, 0.00000e+00, 0.00000e+00, 5.56020e+02, 4.09687e+02],
         [0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 7.39985e+02]]),
  'max Flex breakpoint': [0.1, 5.075, 10.05, 15.025, 20.0],
  'Flex stepsize': 4.876,
  'max Flex stepsize': 4.975,
  '(a, b)': (7, -0.28849178318881075)},
 1: {'Flex breakpoint': [0.098, 4.974, 9.849, 14.725, 19.6],
  'Flex Cost breakpoint': array([[ 1.08400e+00,  1.00000e-02, -8.50000e-02, -1.41000e-01,
          -1.80000e-01],
         [ 0.00000e+00,  5.50300e+01,  2.77330e+01,  2.29720e+01,
           2.01700e+01],
         [ 0.00000e+00,  0.00000e+00,  1.08894e+02,  6.40710e+01,
           5.47770e+01],
         [ 0.00000e

In [8]:
def get_season(date: datetime.datetime):

    season_dict = {"Winter": [1, 2, 12], 
                   "Spring": [3, 4, 5], 
                   "Summer": [6, 7, 8], 
                   "Autumn": [9, 10, 11]}

    month = date.month
    for season, months in season_dict.items():
        if month in months:
            return season

In [11]:
market_name = "mFRR_capacity"
price_area = ["DK1"]
start_dates = {"Summer": datetime.datetime(2023, 7, 15, 0, 0, 0), 
               "Winter": datetime.datetime(2024, 1, 18, 0, 0, 0), 
               "Autumn": datetime.datetime(2023, 10, 15, 0, 0, 0), 
               "Spring": datetime.datetime(2024, 4, 15, 0, 0, 0)}

end_dates = {key: date + datetime.timedelta(days=1) 
             for key, date in start_dates.items()}
timezone = "UTC"
market_data_input = {}

for key in start_dates.keys():
    start_date = start_dates[key]
    end_date = end_dates[key]
    season = get_season(start_date)
    data = energinet_api_wrapper(market_name, price_area, start_date, end_date, timezone)
    
    market_price = data.mFRR_UpPriceDKK.values
    activation_signal = np.array([1 for i in range(len(data))])
    
    market_data_input[key] = [Market_Data(market_price=market_price, 
                                         activation_signal=activation_signal, 
                                         season=season)]
write_pickle(market_data_input, os.path.join(optimization_input_path, "market_data_input.pickle"))

Incorrect url: https://api.energidataservice.dk/dataset/mFRRCapacityMarket/?sort=HourUTC&start=2023-07-15T00:00&end=2023-07-16T00:00&timezone=UTC&filter={"PriceArea":["DK1"]}


AttributeError: 'DataFrame' object has no attribute 'mFRR_UpPriceDKK'

In [ ]:
market_name = "mFRR_capacity"
price_area = ["DK1"]
start_dates = {"High_prices": [datetime.datetime(x, y, z, 0, 0, 0) 
                               for x, y, z in zip([2024]*8 + [2023]*2, [1]*3 + [2] + [8]*5 + [9], [7, 12, 15, 9, 26, 27, 28, 29, 30, 1])], 
               "Med_prices": [datetime.datetime(x, y, z, 0, 0, 0) 
                               for x, y, z in zip([2023]*5 + [2024]*5, [11]*3 + [12]*2 + [1]*5 , [6, 16, 29, 1, 4, 3, 9, 16, 17, 19])], 
               "Low_prices": [datetime.datetime(x, y, z, 0, 0, 0) 
                               for x, y, z in zip([2024]*10, [4]*4 + [6]*3 + [7]*3, [4, 24, 25, 26, 6, 14, 20, 9, 16, 31])]
              }

end_dates = {key: [date + datetime.timedelta(days=1) for date in dates] 
             for key, dates in start_dates.items()}
timezone = "UTC"
market_data_input = {}

for key, dates in start_dates.items():
    market_data_input[key] = []
    for index, start_date in enumerate(dates):
        season = get_season(start_date)
        end_date = end_dates[key][index]
        data = energinet_api_wrapper(market_name, price_area, start_date, end_date, timezone)
        
        market_price = data.mFRR_UpPriceDKK.values
        activation_signal = np.array([1 for i in range(len(data))])
    
        market_data_input[key].append(Market_Data(market_price=market_price, 
                                                  activation_signal=activation_signal, 
                                                  season=season)
                                     )
write_pickle(market_data_input, os.path.join(optimization_input_path, "price_scenarios.pickle"))

In [ ]:
market_data_input = read_pickle(os.path.join(optimization_input_path, "price_scenarios.pickle"))

In [ ]:
market_data_input.keys()

In [10]:
import pandas as pd
from matplotlib import pyplot as plt

In [ ]:
xkcd_palette = ['xkcd:black', 'xkcd:blue', 'xkcd:orange', 'xkcd:green', 'xkcd:red', 'xkcd:purple', 'xkcd:grey']

In [ ]:
k = 0
fig, ax = plt.subplots(figsize=(5, 3))
price_mean = [0, 1, 2]
for price_scenario, market_objects in market_data_input.items():
    if price_scenario == "Med_prices":
        price_scenario = "Medium_prices"
    price_df = pd.DataFrame([market_object.market_price for market_object in market_objects])
    price_df.mean().plot(linewidth=2, color=xkcd_palette[k], label=price_scenario.replace("_", " "), figsize=(5, 3), zorder=2, ax=ax)
    # price_df.transpose().plot(linewidth=2, color=xkcd_palette[k], figsize=(5, 3), zorder=1, alpha=0.2, ax=ax, legend=False)
    k = k+1
plt.yscale("log")
plt.xlabel("Hour of day", fontsize="x-large")
plt.ylabel(r"Mean $\mathbf{\lambda^{M}}$, DKK/MWh", fontsize="x-large")
plt.legend(ncols=2, fontsize=12, loc="upper center")
plt.xticks([4*(i) for i in range(7)], fontsize="large")
plt.yticks(fontsize="large")
plt.xlim([-1, 25])
plt.ylim([1, 30000])
plt.grid(zorder=0, alpha=0.6)
plt.savefig(os.path.join(cwd, "images", "old_price_scenarios.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
k = 0
fig, ax = plt.subplots(figsize=(5, 3))

# Placeholder for mean line handles and labels
price_mean_lines = []
price_mean_labels = []

for price_scenario, market_objects in market_data_input.items():
    if price_scenario == "Med_prices":
        price_scenario = "Medium_prices"
    
    # DataFrame from market prices
    price_df = pd.DataFrame([market_object.market_price for market_object in market_objects])
    
    # Plot the mean line and capture the handle
    mean_line, = ax.plot(
        price_df.mean(),  # Explicitly use ax.plot to capture Line2D
        linewidth=2,
        color=xkcd_palette[k],
        label=price_scenario.replace("_", " "),
        zorder=2
    )
    price_mean_lines.append(mean_line)  # Store the Line2D handle
    price_mean_labels.append(price_scenario.replace("_", " "))  # Store label
    
    # Plot individual lines with transparency
    price_df.transpose().plot(
        linewidth=2,
        color=xkcd_palette[k],
        zorder=1,
        alpha=0.2,
        ax=ax,
        legend=False
    )
    k += 1

# Log scale for y-axis
plt.yscale("log")
plt.xlabel("Hour of day", fontsize="x-large")
plt.ylabel(r"$\mathbf{\lambda^{M}}$, DKK/MWh", fontsize="x-large")

# Custom legend for mean lines only
ax.legend(
    handles=price_mean_lines,  # Use stored Line2D handles
    labels=price_mean_labels,  # Use stored labels
    fontsize=12,
    loc="upper center",
    ncol=2
)

# Formatting ticks and grid
plt.xticks([4 * i for i in range(7)], fontsize="large")
plt.yticks(fontsize="large")
plt.xlim([-1, 25])
plt.ylim([1, 100000])
plt.grid(zorder=0, alpha=0.6)

# Save and show the plot
output_path = os.path.join(os.getcwd(), "images", "price_scenarios.pdf")
plt.savefig(os.path.join(cwd, "images", "price_scenarios.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
k = 0
fig, ax = plt.subplots(figsize=(10, 3))

# Placeholder for mean line handles and labels
price_mean_lines = []
price_mean_labels = []

for price_scenario, market_objects in market_data_input.items():
    if price_scenario == "Med_prices":
        price_scenario = "Medium_prices"
    
    # DataFrame from market prices
    price_df = pd.DataFrame([market_object.market_price for market_object in market_objects])
    
    # Plot the mean line and capture the handle
    mean_line, = ax.plot(
        price_df.mean(),  # Explicitly use ax.plot to capture Line2D
        linewidth=2,
        color=xkcd_palette[k],
        label=price_scenario.replace("_", " "),
        zorder=2
    )
    price_mean_lines.append(mean_line)  # Store the Line2D handle
    price_mean_labels.append(price_scenario.replace("_", " "))  # Store label
    
    # Plot individual lines with transparency
    price_df.transpose().plot(
        linewidth=2,
        color=xkcd_palette[k],
        zorder=1,
        alpha=0.2,
        ax=ax,
        legend=False
    )
    k += 1

# Log scale for y-axis
plt.yscale("log")
plt.xlabel("Hour of day", fontsize=20)
plt.ylabel(r"$\mathbf{\lambda^{\text{M}}}$, DKK/MWh " + "\n" + "(log-scale)", fontsize=20)

# Custom legend for mean lines only
ax.legend(
    handles=price_mean_lines,  # Use stored Line2D handles
    labels=price_mean_labels,  # Use stored labels
    fontsize=16,
    loc="upper right",
    ncol=3
)

# Formatting ticks and grid
plt.xticks([4 * i for i in range(7)], fontsize=16)
plt.yticks(fontsize=16)
plt.xlim([-1, 25])
plt.ylim([1, 40000])
plt.grid(linestyle="--", zorder=0, alpha=0.6)

# Save and show the plot
output_path = os.path.join(os.getcwd(), "images", "price_scenarios.pdf")
plt.savefig(os.path.join(cwd, "images", "price_scenarios_v2.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
prosumer_data = prosumer_cluster["Winter"]

In [ ]:
df = pd.DataFrame(prosumer_data.max_flexibility_profile, columns=["max DSF MWh"])

In [ ]:
df.plot(figsize=(5, 3), color=xkcd_palette[2], legend=False, linewidth=2)
plt.xlabel("Hour of day", fontsize="x-large")
plt.ylabel(r"$\mathbf{\overline{x}}$, MWh", fontsize="x-large")
plt.xticks([4*(i) for i in range(7)], fontsize="large")
plt.yticks(fontsize="large")
plt.xlim([-1, 25])
plt.ylim([1, 15])
plt.grid(zorder=0, alpha=0.6)
plt.savefig(os.path.join(cwd, "images", "max_flexibility.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
df.plot(figsize=(10, 3), color=xkcd_palette[2], legend=False, linewidth=2)
plt.xlabel("Hour of day", fontsize=20)
plt.ylabel(r"$\mathbf{\overline{x}}$, MWh", fontsize=20)
plt.xticks([4*(i) for i in range(7)], fontsize=16)
plt.yticks(fontsize=16)
plt.xlim([-1, 25])
plt.ylim([1, 15])
plt.grid(linestyle="--", zorder=0, alpha=0.6)
plt.savefig(os.path.join(cwd, "images", "max_flexibility_v2.pdf"), bbox_inches="tight")
plt.show()